# N-ATLaS Training Input Boundary & Formatter Smoke Test

This notebook verifies that `NAtlasFormatter` converts canonical Foundry `DatasetRecord` objects into conversational messages (`user` -> `input`, `assistant` -> `target`) and applies the official `NCAIR1/N-ATLaS` chat template (`tokenize=False`, `add_generation_prompt=False`) on actual hardware/environment.

In [1]:
# Install dependencies if needed in Kaggle environment
!pip install -q -U transformers accelerate pydantic huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 3.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 80.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.1/487.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 23.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 35.0 MB/s eta 0:00:00


In [5]:
import sys
from pathlib import Path

# If running inside cloned repository or root
repo_root = Path.cwd()
if (repo_root / "foundry").exists() and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dataclasses import dataclass
from pydantic import BaseModel, Field

try:
    from api.schemas.dataset import DatasetRecord
    from foundry.training.natlas_formatter import NAtlasFormatter, FormattedTrainingExample
    print("Successfully imported DatasetRecord and NAtlasFormatter from repository.")
except ImportError:
    print("Falling back to self-contained classes for standalone Kaggle run:")
    
    class DatasetRecord(BaseModel):
        id: str = Field(..., min_length=1)
        input: str = Field(..., min_length=1)
        target: str = Field(..., min_length=1)
        language_code: str = Field(..., min_length=2, max_length=8)

    @dataclass(frozen=True)
    class FormattedTrainingExample:
        record_id: str
        language_code: str
        messages: list[dict[str, str]]
        formatted_text: str

    class NAtlasFormatter:
        def __init__(self, tokenizer):
            self.tokenizer = tokenizer

        def _validate_record(self, record: DatasetRecord) -> None:
            if not record.input.strip():
                raise ValueError("DatasetRecord.input cannot be empty")
            if not record.target.strip():
                raise ValueError("DatasetRecord.target cannot be empty")
            if not record.language_code.strip():
                raise ValueError("DatasetRecord.language_code cannot be empty")

        def build_messages(self, record: DatasetRecord) -> list[dict[str, str]]:
            return [
                {"role": "user", "content": record.input},
                {"role": "assistant", "content": record.target},
            ]

        def format(self, record: DatasetRecord) -> FormattedTrainingExample:
            self._validate_record(record)
            messages = self.build_messages(record)
            formatted_text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            return FormattedTrainingExample(
                record_id=record.id,
                language_code=record.language_code,
                messages=messages,
                formatted_text=formatted_text,
            )
    print("Self-contained NAtlasFormatter ready.")

Falling back to self-contained classes for standalone Kaggle run:
Self-contained NAtlasFormatter ready.


In [8]:
# Authenticate hugging face
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

print("Hugging Face authentication completed.")

Hugging Face authentication completed.


In [9]:
# Load N-ATLaS tokenizer from Hugging Face
from transformers import AutoTokenizer

MODEL_NAME = "NCAIR1/N-ATLaS"
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Chat template defined:", tokenizer.chat_template is not None)

Loading tokenizer for NCAIR1/N-ATLaS...


config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

Tokenizer class: TokenizersBackend
Chat template defined: True


In [10]:
# Create sample Igala DatasetRecord
record = DatasetRecord(
    id="formatter-smoke-001",
    input="How are you?",
    target="Abe ele?",
    language_code="igl",
)

formatter = NAtlasFormatter(tokenizer)
result = formatter.format(record)

print("Record ID:", result.record_id)
print("Language:", result.language_code)
print("Messages:")
print(result.messages)
print()
print("Formatted text:")
print(result.formatted_text)

Record ID: formatter-smoke-001
Language: igl
Messages:
[{'role': 'user', 'content': 'How are you?'}, {'role': 'assistant', 'content': 'Abe ele?'}]

Formatted text:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

How are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Abe ele?<|eot_id|>


In [11]:
# Inspect tokenization count and special tokens
token_output = tokenizer(result.formatted_text)
token_ids = token_output["input_ids"]
print("Token count:", len(token_ids))
print("Token IDs:", token_ids)
print("Decoded representation with tokens:")
for tid in token_ids:
    print(f"  {tid} -> {repr(tokenizer.decode([tid]))}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Token count: 45
Token IDs: [128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 128009, 128006, 882, 128007, 271, 4438, 527, 499, 30, 128009, 128006, 78191, 128007, 271, 32, 1395, 10732, 30, 128009]
Decoded representation with tokens:
  128000 -> '<|begin_of_text|>'
  128000 -> '<|begin_of_text|>'
  128006 -> '<|start_header_id|>'
  9125 -> 'system'
  128007 -> '<|end_header_id|>'
  271 -> '\n\n'
  38766 -> 'Cut'
  1303 -> 'ting'
  33025 -> ' Knowledge'
  2696 -> ' Date'
  25 -> ':'
  6790 -> ' December'
  220 -> ' '
  2366 -> '202'
  18 -> '3'
  198 -> '\n'
  15724 -> 'Today'
  2696 -> ' Date'
  25 -> ':'
  220 -> ' '
  1627 -> '26'
  10263 -> ' Jul'
  220 -> ' '
  2366 -> '202'
  19 -> '4'
  271 -> '\n\n'
  128009 -> '<|eot_id|>'
  128006 -> '<|start_header_id|>'
  882 -> 'user'
  128007 -> '<|end_header_id|>'
  271 -> '\n\n'
  4438 -> 'How'
  527 -> ' are'
  499 -> ' you'
  30 -> '?

In [12]:
# Multi-language verification test (Igala, Yoruba, Igbo, Hausa)
records = [
    DatasetRecord(id="rec-igl", input="Good morning", target="Ólódù", language_code="igl"),
    DatasetRecord(id="rec-yor", input="Good morning", target="Ẹ ku aarọ", language_code="yor"),
    DatasetRecord(id="rec-ibo", input="Good morning", target="Ụtụtụ ọma", language_code="ibo"),
    DatasetRecord(id="rec-hau", input="Good morning", target="Ina kwana", language_code="hau"),
]

for rec in records:
    res = formatter.format(rec)
    print(f"[{res.language_code}] {res.record_id} -> tokens: {len(tokenizer(res.formatted_text)['input_ids'])}")
    print("   Formatted:", repr(res.formatted_text))
    print("-" * 60)

[igl] rec-igl -> tokens: 43
   Formatted: '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nGood morning<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nÓlódù<|eot_id|>'
------------------------------------------------------------
[yor] rec-yor -> tokens: 45
   Formatted: '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nGood morning<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nẸ ku aarọ<|eot_id|>'
------------------------------------------------------------
[ibo] rec-ibo -> tokens: 48
   Formatted: '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nGood morning<|eot_id